# IS 507 Data Preparation and Profiling

This notebook uses the two single-file Parquet datasets shared with the team:

- `data/asi_opensource_pod_hourly_day0_29.parquet`
- `data/asi_opensource_job_execution_summary.parquet`

The workflow is intentionally simple:

1. Validate file completeness and inspect missing or exceptional values.
2. Define transparent query-time adjustments without rewriting the raw files.
3. Produce compact field, numeric, and categorical profile tables.

The original 720 hourly files may remain on the data owner's machine, but this notebook never requires them.


In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "MID" / "data").exists():
    ROOT = ROOT / "MID"

DATA_DIR = ROOT / "data"
POD_PARQUET = DATA_DIR / "asi_opensource_pod_hourly_day0_29.parquet"
JOB_PARQUET = DATA_DIR / "asi_opensource_job_execution_summary.parquet"

assert POD_PARQUET.exists(), f"Missing pod-hourly file: {POD_PARQUET}"
assert JOB_PARQUET.exists(), f"Missing execution-summary file: {JOB_PARQUET}"

con = duckdb.connect()
con.execute("SET threads=4")
con.execute("SET memory_limit='8GB'")
TEMP_DIR = DATA_DIR / ".duckdb_tmp"
TEMP_DIR.mkdir(parents=True, exist_ok=True)
con.execute(f"SET temp_directory='{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order=false")

con.execute(f'''CREATE OR REPLACE VIEW pod_raw AS
                SELECT * FROM read_parquet('{POD_PARQUET.as_posix()}')''')
con.execute(f'''CREATE OR REPLACE VIEW job_raw AS
                SELECT * FROM read_parquet('{JOB_PARQUET.as_posix()}')''')

# Quick diagnostic scope. Expand these values only for a deliberate full profile.
POD_PROFILE_DAYS = [0]
POD_PROFILE_HOURS = [0]
JOB_PROFILE_ROW_LIMIT = 1_000_000

profile_days_sql = ", ".join(map(str, POD_PROFILE_DAYS))
profile_hours_sql = ", ".join(map(str, POD_PROFILE_HOURS))
con.execute(f'''CREATE OR REPLACE VIEW pod_profile_raw AS
                SELECT * FROM pod_raw
                WHERE day IN ({profile_days_sql})
                  AND hour IN ({profile_hours_sql})''')
con.execute(f'''CREATE OR REPLACE VIEW job_profile_raw AS
                SELECT * FROM job_raw LIMIT {JOB_PROFILE_ROW_LIMIT}''')

print("Pod-hourly:", POD_PARQUET)
print("Execution summary:", JOB_PARQUET)
print("Quick pod profile:", POD_PROFILE_DAYS, POD_PROFILE_HOURS)
print("Quick summary rows:", f"{JOB_PROFILE_ROW_LIMIT:,}")


Pod-hourly: c:\Users\ethan\Desktop\UIUC26Fall\IS507\MID\data\asi_opensource_pod_hourly_day0_29.parquet
Execution summary: c:\Users\ethan\Desktop\UIUC26Fall\IS507\MID\data\asi_opensource_job_execution_summary.parquet
Quick pod profile: [0] [0]
Quick summary rows: 1,000,000


## 1. Completeness and missing-value analysis

File metadata and day/hour coverage validate the complete datasets. To keep the notebook interactive, missing-value and profile tables use Day 0 / Hour 0 for pod data and the first one million execution-summary rows. These quick profiles diagnose structure and data-quality issues; they are not population estimates.


In [2]:
EXPECTED_POD_ROWS = 842_390_418
EXPECTED_JOB_ROWS = 40_522_321

def parquet_metadata(path):
    parquet = pq.ParquetFile(path)
    return {
        "file": path.name,
        "size_gib": path.stat().st_size / 1024**3,
        "rows": parquet.metadata.num_rows,
        "row_groups": parquet.metadata.num_row_groups,
        "columns": parquet.metadata.num_columns,
    }

file_inventory = pd.DataFrame([
    parquet_metadata(POD_PARQUET),
    parquet_metadata(JOB_PARQUET),
])
display(file_inventory)

assert int(file_inventory.loc[0, "rows"]) == EXPECTED_POD_ROWS
assert int(file_inventory.loc[1, "rows"]) == EXPECTED_JOB_ROWS

pod_coverage = con.execute('''
SELECT
    count(*) AS rows,
    min(day) AS min_day,
    max(day) AS max_day,
    count(DISTINCT day) AS distinct_days,
    min(hour) AS min_hour,
    max(hour) AS max_hour,
    count(DISTINCT (day, hour)) AS distinct_day_hours
FROM pod_raw
''').df()
display(pod_coverage)

coverage = pod_coverage.iloc[0]
assert int(coverage["rows"]) == EXPECTED_POD_ROWS
assert (int(coverage["min_day"]), int(coverage["max_day"])) == (0, 29)
assert int(coverage["distinct_days"]) == 30
assert (int(coverage["min_hour"]), int(coverage["max_hour"])) == (0, 23)
assert int(coverage["distinct_day_hours"]) == 720
print("Completeness checks passed.")


,file,size_gib,rows,row_groups,columns
0,asi_opensource_pod_hourly_day0_29.parquet,33.475704,842390418,844,25
1,asi_opensource_job_execution_summary.parquet,1.106686,40522321,346,14


,rows,min_day,max_day,distinct_days,min_hour,max_hour,distinct_day_hours
0,842390418,0,29,30,0,23,720


Completeness checks passed.


In [3]:
pod_quality = con.execute('''
SELECT
    count(*) AS row_count,
    count(DISTINCT pod_id) AS unique_pods,
    count(*) FILTER (WHERE workload_id IS NULL) AS missing_workload_id,
    count(*) FILTER (WHERE workload_id = '') AS blank_workload_id,
    count(*) FILTER (WHERE avg_gpu_sm_util IS NULL) AS missing_avg_gpu_sm_util,
    count(*) FILTER (WHERE schedule_delay_sec IS NULL) AS missing_schedule_delay,
    count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec IS NULL) AS missing_ready_delay,
    count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay
FROM pod_profile_raw
''').df()

job_quality = con.execute('''
SELECT
    count(*) AS row_count,
    count(DISTINCT pod_id) AS unique_pods,
    count(*) FILTER (WHERE workload_id IS NULL) AS missing_workload_id,
    count(*) FILTER (WHERE workload_id = '') AS blank_workload_id,
    count(*) FILTER (WHERE duration_hours IS NULL) AS missing_duration,
    count(*) FILTER (WHERE duration_hours < 0) AS negative_duration,
    count(*) FILTER (WHERE schedule_delay_sec IS NULL) AS missing_schedule_delay,
    count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec IS NULL) AS missing_ready_delay,
    count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay
FROM job_profile_raw
''').df()

def quality_table(frame, dataset):
    row = frame.iloc[0]
    total = int(row["row_count"])
    return pd.DataFrame([
        {
            "dataset": dataset,
            "metric": metric,
            "count": int(value),
            "percent_of_rows": 100.0 * float(value) / total,
        }
        for metric, value in row.items()
    ])

pod_scope_label = f"pod_hourly (days {POD_PROFILE_DAYS}, hours {POD_PROFILE_HOURS})"
job_scope_label = f"execution_summary (first {JOB_PROFILE_ROW_LIMIT:,} rows)"
quality_report = pd.concat([
    quality_table(pod_quality, pod_scope_label),
    quality_table(job_quality, job_scope_label),
], ignore_index=True)
display(quality_report)


,dataset,metric,count,percent_of_rows
0,"pod_hourly (days [0], hours [0])",row_count,1179029,100.000000
1,"pod_hourly (days [0], hours [0])",unique_pods,1179028,99.999915
2,"pod_hourly (days [0], hours [0])",missing_workload_id,273629,23.207996
3,"pod_hourly (days [0], hours [0])",blank_workload_id,0,0.000000
4,"pod_hourly (days [0], hours [0])",missing_avg_gpu_sm_util,1110800,94.213119
5,"pod_hourly (days [0], hours [0])",missing_schedule_delay,24350,2.065259
6,"pod_hourly (days [0], hours [0])",negative_schedule_delay,15,0.001272
7,"pod_hourly (days [0], hours [0])",missing_ready_delay,24350,2.065259
8,"pod_hourly (days [0], hours [0])",negative_ready_delay,334973,28.410921
9,"execution_summary (first 1,000,000 rows)",row_count,1000000,100.000000


### Interpretation rules

- Missing values are not removed globally.
- Literal `Unknown` or `unknown` values remain valid source categories.
- Negative delays are preserved in raw columns but excluded from clean delay fields.
- GPU-utilization missingness should also be evaluated after filtering to `gpu_request > 0`; many pod-hour rows do not request a GPU.
- Extreme duration or utilization values are flagged for study, not automatically deleted.


## 2. Query-time adjustments

These views do not create another large processed file. They preserve every raw column and add clearly named analysis fields.

`workload_id` treatment is intentionally conservative because the team has not chosen a final strategy. Blank strings are normalized to `NULL`, but missing IDs are not imputed and rows are not dropped.


In [4]:
con.execute('''
CREATE OR REPLACE VIEW pod_analysis AS
SELECT
    *,
    nullif(trim(workload_id), '') AS workload_id_clean,
    CASE
        WHEN workload_id IS NULL OR trim(workload_id) = '' THEN 'missing'
        ELSE 'observed'
    END AS workload_id_status,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END
        AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END
        AS ready_delay_sec_clean,
    gpu_request > 0 AS is_gpu_request
FROM pod_raw
''')

con.execute('''
CREATE OR REPLACE VIEW job_analysis AS
SELECT
    *,
    nullif(trim(workload_id), '') AS workload_id_clean,
    CASE
        WHEN workload_id IS NULL OR trim(workload_id) = '' THEN 'missing'
        ELSE 'observed'
    END AS workload_id_status,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END
        AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END
        AS ready_delay_sec_clean
FROM job_raw
''')

con.execute('''
CREATE OR REPLACE VIEW pod_profile_analysis AS
SELECT
    *,
    nullif(trim(workload_id), '') AS workload_id_clean,
    CASE
        WHEN workload_id IS NULL OR trim(workload_id) = '' THEN 'missing'
        ELSE 'observed'
    END AS workload_id_status,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END
        AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END
        AS ready_delay_sec_clean,
    gpu_request > 0 AS is_gpu_request
FROM pod_profile_raw
''')

con.execute('''
CREATE OR REPLACE VIEW job_profile_analysis AS
SELECT
    *,
    nullif(trim(workload_id), '') AS workload_id_clean,
    CASE
        WHEN workload_id IS NULL OR trim(workload_id) = '' THEN 'missing'
        ELSE 'observed'
    END AS workload_id_status,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END
        AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END
        AS ready_delay_sec_clean
FROM job_profile_raw
''')

preprocessing_policy = pd.DataFrame([
    ["workload_id", "Trim blanks to NULL; do not impute; keep all rows"],
    ["schedule_delay_sec", "Keep raw; add non-negative *_clean field"],
    ["ready_delay_sec", "Keep raw; add non-negative *_clean field"],
    ["Unknown categories", "Keep as documented categories"],
    ["numeric outliers", "Keep and profile before any research-specific rule"],
])
preprocessing_policy.columns = ["field", "current_rule"]
display(preprocessing_policy)


,field,current_rule
0,workload_id,Trim blanks to NULL; do not impute; keep all rows
1,schedule_delay_sec,Keep raw; add non-negative *_clean field
2,ready_delay_sec,Keep raw; add non-negative *_clean field
3,Unknown categories,Keep as documented categories
4,numeric outliers,Keep and profile before any research-specific ...


In [5]:
workload_id_coverage = con.execute('''
SELECT
    'pod quick profile' AS dataset,
    count(*) AS row_count,
    count(*) FILTER (WHERE workload_id_clean IS NOT NULL) AS rows_with_workload_id,
    count(DISTINCT workload_id_clean) AS unique_workloads
FROM pod_profile_analysis
UNION ALL
SELECT
    'summary quick profile',
    count(*),
    count(*) FILTER (WHERE workload_id_clean IS NOT NULL),
    count(DISTINCT workload_id_clean)
FROM job_profile_analysis
''').df()
workload_id_coverage["coverage_percent"] = (
    100 * workload_id_coverage["rows_with_workload_id"]
    / workload_id_coverage["row_count"]
)
display(workload_id_coverage)


,dataset,row_count,rows_with_workload_id,unique_workloads,coverage_percent
0,pod quick profile,1179029,905400,55477,76.792004
1,summary quick profile,1000000,119647,35801,11.964700


### Pending `workload_id` decision

Recommended default: keep missing IDs for single-table analyses. For a workload-level aggregation or join, use only non-missing `workload_id_clean` values and report the retained-row and retained-workload coverage.

Do not silently replace a missing workload ID with `pod_id`: that changes the analytical grain. If the team later needs a fallback grouping key, create a separate, explicitly named field and document its meaning.


## 3. Data profile tables

DuckDB's `SUMMARIZE` command returns one row per field with type, non-null count, null percentage, approximate unique count, range, average, standard deviation, and quartiles where applicable. Both tables use the quick-profile scope defined in the setup cell.


In [6]:
pod_column_profile = con.execute(
    "SUMMARIZE SELECT * FROM pod_profile_analysis"
).df()
pod_column_profile.insert(0, "dataset", pod_scope_label)

job_column_profile = con.execute(
    "SUMMARIZE SELECT * FROM job_profile_analysis"
).df()
job_column_profile.insert(0, "dataset", job_scope_label)

column_profile = pd.concat(
    [pod_column_profile, job_column_profile],
    ignore_index=True,
)
display(column_profile)


,dataset,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,"pod_hourly (days [0], hours [0])",pod_id,VARCHAR,00001dcdb04cc66c4a4748fb8e9e9989,ffffff76477ac62e023598362cb1fbb4,1451811,NaN,NaN,NaN,NaN,NaN,1179029,0.00
1,"pod_hourly (days [0], hours [0])",workload_id,VARCHAR,00004bedaf9d9cc3007aa09554d799f3,ffff314bbc8e3d2a204e522b8d499a37,56785,NaN,NaN,NaN,NaN,NaN,1179029,23.21
2,"pod_hourly (days [0], hours [0])",server_id,VARCHAR,0000e52b595d367dd13eebf63ab55737,fffc99e2aeb909eebf98582d754c7c22,31692,NaN,NaN,NaN,NaN,NaN,1179029,0.00
3,"pod_hourly (days [0], hours [0])",cluster_id,VARCHAR,1f28e063c4e5e1bc48ff55269617dd2f,da8d7fb8796ea962e14eef38624d8059,12,NaN,NaN,NaN,NaN,NaN,1179029,0.00
4,"pod_hourly (days [0], hours [0])",state_public,VARCHAR,Unknown,Unknown,1,NaN,NaN,NaN,NaN,NaN,1179029,0.00
5,"pod_hourly (days [0], hours [0])",gpu_spec_public,VARCHAR,A10,XPU-E,13,NaN,NaN,NaN,NaN,NaN,1179029,0.00
6,"pod_hourly (days [0], hours [0])",server_gpu_count,BIGINT,1,16,4,4.094347128018056,4.623767964707705,1,2,8,1179029,0.00
7,"pod_hourly (days [0], hours [0])",server_cpu_capacity_cores,BIGINT,128,192,2,171.12457115134572,30.004077726971925,128,192,192,1179029,0.00
8,"pod_hourly (days [0], hours [0])",priority_class,VARCHAR,HP,Other,3,NaN,NaN,NaN,NaN,NaN,1179029,0.00
9,"pod_hourly (days [0], hours [0])",job_type_public,VARCHAR,dev,unknown,5,NaN,NaN,NaN,NaN,NaN,1179029,0.00


In [7]:
def numeric_profile(table, dataset, columns):
    expressions = ["count(*) AS total_rows"]
    for column in columns:
        expressions.extend([
            f'count("{column}") AS "{column}__non_null"',
            f'min("{column}") AS "{column}__min"',
            f'approx_quantile("{column}", 0.25) AS "{column}__q25"',
            f'approx_quantile("{column}", 0.50) AS "{column}__median"',
            f'approx_quantile("{column}", 0.75) AS "{column}__q75"',
            f'approx_quantile("{column}", 0.99) AS "{column}__p99"',
            f'max("{column}") AS "{column}__max"',
            f'avg("{column}") AS "{column}__mean"',
            f'stddev_pop("{column}") AS "{column}__sd"',
        ])
    row = con.execute(
        f"SELECT {', '.join(expressions)} FROM {table}"
    ).df().iloc[0]
    total = int(row["total_rows"])
    records = []
    for column in columns:
        records.append({
            "dataset": dataset,
            "column": column,
            "non_null_count": int(row[f"{column}__non_null"]),
            "non_null_percent": 100 * float(row[f"{column}__non_null"]) / total,
            "min": row[f"{column}__min"],
            "q25": row[f"{column}__q25"],
            "median": row[f"{column}__median"],
            "q75": row[f"{column}__q75"],
            "p99": row[f"{column}__p99"],
            "max": row[f"{column}__max"],
            "mean": row[f"{column}__mean"],
            "sd": row[f"{column}__sd"],
        })
    return pd.DataFrame(records)

pod_numeric = numeric_profile("pod_profile_analysis", pod_scope_label, [
    "gpu_request", "used_gpu_hours", "avg_gpu_sm_util",
    "avg_gpu_mem_gib", "cpu_request_cores",
    "schedule_delay_sec_clean", "ready_delay_sec_clean",
])
job_numeric = numeric_profile("job_profile_analysis", job_scope_label, [
    "gpu_request", "duration_hours",
    "schedule_delay_sec_clean", "ready_delay_sec_clean",
])
numeric_distribution = pd.concat([pod_numeric, job_numeric], ignore_index=True)
display(numeric_distribution)


,dataset,column,non_null_count,non_null_percent,min,q25,median,q75,p99,max,mean,sd
0,"pod_hourly (days [0], hours [0])",gpu_request,1179029,100.000000,0.000000,0.000000,0.000000,0.000000,1.000000,16.000000,0.093638,0.585528
1,"pod_hourly (days [0], hours [0])",used_gpu_hours,1179029,100.000000,0.000000,0.000000,0.000000,0.000000,1.000000,16.000000,0.083299,0.557524
2,"pod_hourly (days [0], hours [0])",avg_gpu_sm_util,68229,5.786881,0.000000,0.473765,4.645599,19.314376,485.119184,855.874529,24.174241,74.507419
3,"pod_hourly (days [0], hours [0])",avg_gpu_mem_gib,85812,7.278192,0.000000,3.121227,12.058248,86.982676,614.717016,1511.879150,54.336328,119.761241
4,"pod_hourly (days [0], hours [0])",cpu_request_cores,1179029,100.000000,0.000000,0.193839,1.000000,3.824983,63.599018,192.000000,4.868031,13.959088
5,"pod_hourly (days [0], hours [0])",schedule_delay_sec_clean,1154664,97.933469,0.000000,0.000000,0.000000,10.000000,3840.000000,86366.000000,246.638891,2799.341791
6,"pod_hourly (days [0], hours [0])",ready_delay_sec_clean,819706,69.523820,0.000000,3.000000,7.000000,44.000000,9571.000000,86393.000000,550.133572,5389.997936
7,"execution_summary (first 1,000,000 rows)",gpu_request,1000000,100.000000,0.010000,1.000000,1.000000,1.000000,8.004772,16.000000,1.214405,1.577362
8,"execution_summary (first 1,000,000 rows)",duration_hours,1000000,100.000000,0.005556,0.129476,0.427899,1.504034,72.829512,1823.994444,5.337620,48.851162
9,"execution_summary (first 1,000,000 rows)",schedule_delay_sec_clean,984511,98.451100,0.000000,0.000000,0.000000,72.000000,8432.000000,86396.000000,454.633527,4528.396946


In [8]:
def top_categories(table, dataset, columns, top_n=10):
    total = con.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
    frames = []
    for column in columns:
        frame = con.execute(f'''            SELECT
                '{dataset}' AS dataset,
                '{column}' AS column,
                coalesce(cast("{column}" AS VARCHAR), '<NULL>') AS value,
                count(*) AS count
            FROM {table}
            GROUP BY "{column}"
            ORDER BY count DESC
            LIMIT {int(top_n)}
        ''').df()
        frame["percent"] = 100 * frame["count"] / total
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

pod_categories = top_categories("pod_profile_analysis", pod_scope_label, [
    "state_public", "gpu_spec_public", "priority_class",
    "job_type_public", "model_type_public", "is_genai_request",
    "ready_status", "workload_id_status",
])
job_categories = top_categories("job_profile_analysis", job_scope_label, [
    "gpu_spec_public", "priority_class", "job_type_public",
    "model_type_public", "is_genai_request", "ready_status",
    "schedule_status", "workload_id_status",
])
categorical_distribution = pd.concat(
    [pod_categories, job_categories],
    ignore_index=True,
)
display(categorical_distribution)


,dataset,column,value,count,percent
0,"pod_hourly (days [0], hours [0])",state_public,Unknown,1179029,100.000000
1,"pod_hourly (days [0], hours [0])",gpu_spec_public,A10,391106,33.171873
2,"pod_hourly (days [0], hours [0])",gpu_spec_public,XPU-B,210873,17.885311
3,"pod_hourly (days [0], hours [0])",gpu_spec_public,L20,198115,16.803234
4,"pod_hourly (days [0], hours [0])",gpu_spec_public,XPU-E,105176,8.920561
...,...,...,...,...,...
62,"execution_summary (first 1,000,000 rows)",schedule_status,true,984559,98.455900
63,"execution_summary (first 1,000,000 rows)",schedule_status,<NULL>,9597,0.959700
64,"execution_summary (first 1,000,000 rows)",schedule_status,false,5844,0.584400
65,"execution_summary (first 1,000,000 rows)",workload_id_status,missing,879790,87.979000


In [ ]:
# Export small, shareable profile tables (not the raw datasets).
PROFILE_DIR = ROOT / "reports" / "data_profile"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

profile_tables = {
    "file_inventory.csv": file_inventory,
    "quality_report.csv": quality_report,
    "workload_id_coverage.csv": workload_id_coverage,
    "column_profile.csv": column_profile,
    "numeric_distribution.csv": numeric_distribution,
    "categorical_distribution.csv": categorical_distribution,
}

export_manifest = []
for filename, table in profile_tables.items():
    output_path = PROFILE_DIR / filename
    table.to_csv(output_path, index=False)
    export_manifest.append({
        "file": output_path.relative_to(ROOT).as_posix(),
        "rows": len(table),
        "columns": len(table.columns),
    })

export_manifest = pd.DataFrame(export_manifest)
display(export_manifest)
print("Profile tables exported to:", PROFILE_DIR)


## Handoff notes

- The integrity checks cover both complete files; the profile tables use the clearly labeled quick scope.
- The six profile CSV files are generated under `reports/data_profile/`.
- Quick-profile percentages must not be reported as full-population estimates.
- Keep filtering, cleaning, and aggregation lazy in DuckDB or Arrow; do not load either full dataset into pandas or R memory.
- Decide the final `workload_id` rule only after selecting the research question and reporting coverage.
- Aggregate each source to a declared grain before joining. Do not raw-join the two large tables.
